In [2]:
# ============================================================
# FRONT END — INSTALLATION & IMPORTS
# ============================================================

!pip -q install gradio

import gradio as gr
import numpy as np
import matplotlib.pyplot as plt# ============================================================
# MATHEMATICAL ENGINE
# ============================================================

def calculate_curving(
    prof1_grades,
    prof2_grades,
    prof3_grades
):
    """
    Performs affine grade normalization.

    y = a*x + b

    a = Target SD / Original SD
    b = Target Mean - a * Original Mean
    """

    # Convert input text into arrays
    groups = {
        "Professor 1": np.array(
            [float(x) for x in prof1_grades.split()],
            dtype=float
        ),

        "Professor 2": np.array(
            [float(x) for x in prof2_grades.split()],
            dtype=float
        ),

        "Professor 3": np.array(
            [float(x) for x in prof3_grades.split()],
            dtype=float
        )
    }

    # Calculate original statistics
    statistics = {}

    for name, grades in groups.items():

        statistics[name] = {
            "grades": grades,
            "mean": np.mean(grades),
            "std": np.std(grades)
        }

    # Common target values
    target_mean = np.mean([
        statistics[name]["mean"]
        for name in statistics
    ])

    target_std = np.mean([
        statistics[name]["std"]
        for name in statistics
    ])

    results = {}

    # Apply normalization
    for name, data in statistics.items():

        mean = data["mean"]
        std = data["std"]

        if std == 0:
            raise ValueError(
                f"{name} has zero standard deviation."
            )

        # Scaling factor
        a = target_std / std

        # Shift factor
        b = target_mean - a * mean

        adjusted = a * data["grades"] + b

        results[name] = {
            "original": data["grades"],
            "adjusted": adjusted,
            "original_mean": mean,
            "original_std": std,
            "adjusted_mean": np.mean(adjusted),
            "adjusted_std": np.std(adjusted),
            "a": a,
            "b": b
        }

    return results, target_mean, target_std# ============================================================
# FRONT-END PROCESSING FUNCTION
# ============================================================

def run_project(
    professor1,
    professor2,
    professor3
):

    try:

        results, target_mean, target_std = calculate_curving(
            professor1,
            professor2,
            professor3
        )

        # ----------------------------------------------------
        # CREATE OUTPUT TABLE
        # ----------------------------------------------------

        table = []

        for name, data in results.items():

            table.append([
                name,
                f"{data['original_mean']:.2f}",
                f"{data['original_std']:.2f}",
                f"{data['adjusted_mean']:.2f}",
                f"{data['adjusted_std']:.2f}",
                f"{data['a']:.4f}",
                f"{data['b']:.4f}"
            ])

        # ----------------------------------------------------
        # CREATE CHART
        # ----------------------------------------------------

        names = list(results.keys())

        original_means = [
            results[n]["original_mean"]
            for n in names
        ]

        adjusted_means = [
            results[n]["adjusted_mean"]
            for n in names
        ]

        original_std = [
            results[n]["original_std"]
            for n in names
        ]

        adjusted_std = [
            results[n]["adjusted_std"]
            for n in names
        ]

        # Create figure
        fig, axes = plt.subplots(
            1, 2,
            figsize=(12, 5)
        )

        # ----------------------------------------------------
        # MEAN CHART
        # ----------------------------------------------------

        x = np.arange(len(names))
        width = 0.35

        axes[0].bar(
            x - width/2,
            original_means,
            width,
            label="Original"
        )

        axes[0].bar(
            x + width/2,
            adjusted_means,
            width,
            label="Adjusted"
        )

        axes[0].axhline(
            target_mean,
            linestyle="--",
            label="Target Mean"
        )

        axes[0].set_title(
            "Mean Comparison"
        )

        axes[0].set_ylabel(
            "Mean Grade"
        )

        axes[0].set_xticks(x)
        axes[0].set_xticklabels(names)
        axes[0].legend()

        # ----------------------------------------------------
        # STANDARD DEVIATION CHART
        # ----------------------------------------------------

        axes[1].bar(
            x - width/2,
            original_std,
            width,
            label="Original"
        )

        axes[1].bar(
            x + width/2,
            adjusted_std,
            width,
            label="Adjusted"
        )

        axes[1].axhline(
            target_std,
            linestyle="--",
            label="Target SD"
        )

        axes[1].set_title(
            "Standard Deviation Comparison"
        )

        axes[1].set_ylabel(
            "Standard Deviation"
        )

        axes[1].set_xticks(x)
        axes[1].set_xticklabels(names)
        axes[1].legend()

        plt.tight_layout()

        # ----------------------------------------------------
        # MATHEMATICAL EXPLANATION
        # ----------------------------------------------------

        explanation = f"""
## Mathematical Result

### Target values

**Target Mean:** `{target_mean:.4f}`

**Target Standard Deviation:** `{target_std:.4f}`

### Transformation

For each professor:

**y = a × x + b**

where

**a = Target SD / Original SD**

**b = Target Mean − a × Original Mean**

The transformation places all three groups on the same
statistical scale.
"""

        return table, explanation, fig

    except Exception as e:

        return (
            [["Error", str(e)]],
            "Please check your input grades.",
            None
        )# ============================================================
# GRADIO FRONT END
# ============================================================

with gr.Blocks(
    title="Student Grade Curving System"
) as demo:

    # --------------------------------------------------------
    # HEADER
    # --------------------------------------------------------

    gr.Markdown(
        """
        # 🎓 Student Grade Curving System

        ### B.Tech Mathematics Mini Project — A8

        **Purpose:** Normalize grades from three professors
        whose examinations have different difficulty levels.

        **Mathematical Model:**

        `y = a × x + b`

        The system matches the target **mean** and
        **standard deviation**.
        """
    )

    # --------------------------------------------------------
    # INPUT SECTION
    # --------------------------------------------------------

    gr.Markdown(
        """
        ## 📥 Enter Student Grades

        Enter grades separated by spaces.

        Example:

        `45 50 55 60 65`
        """
    )

    with gr.Row():

        professor1 = gr.Textbox(
            label="Professor 1 Grades",
            value="45 50 55 60 65",
            placeholder="45 50 55 60 65"
        )

        professor2 = gr.Textbox(
            label="Professor 2 Grades",
            value="55 60 65 70 75",
            placeholder="55 60 65 70 75"
        )

        professor3 = gr.Textbox(
            label="Professor 3 Grades",
            value="35 45 55 65 75",
            placeholder="35 45 55 65 75"
        )

    # --------------------------------------------------------
    # BUTTON
    # --------------------------------------------------------

    calculate_button = gr.Button(
        "🚀 Calculate Grade Curving",
        variant="primary"
    )

    # --------------------------------------------------------
    # RESULTS
    # --------------------------------------------------------

    gr.Markdown(
        "## 📊 Results"
    )

    result_table = gr.Dataframe(
        headers=[
            "Professor",
            "Original Mean",
            "Original SD",
            "Adjusted Mean",
            "Adjusted SD",
            "Scaling Factor (a)",
            "Shift Factor (b)"
        ],
        label="Grade Curving Results",
        interactive=False
    )

    explanation = gr.Markdown()

    chart = gr.Plot(
        label="Statistical Comparison"
    )

    # --------------------------------------------------------
    # CONNECT BUTTON
    # --------------------------------------------------------

    calculate_button.click(
        fn=run_project,
        inputs=[
            professor1,
            professor2,
            professor3
        ],
        outputs=[
            result_table,
            explanation,
            chart
        ]
    )


# ============================================================
# LAUNCH
# ============================================================

demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cf5582002114cbc86c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
